# 04 - DistilBERT · `product`

**Support Ticket Triage** - Notebook 04 of 6

Fine-tunes `distilbert-base-uncased` on the **product** task (9 classes) and beats,
or fails to beat, the Notebook 03 baseline.

| Model | Validation macro-F1 |
|---|---|
| Stratified dummy | 0.1103 |
| TF-IDF + LogReg | **0.7748** |
| DistilBERT | *this notebook* |

**A number below 0.7748 is a real result, not a failure to hide.** It would say the
lexical signal in these narratives is already saturated by bag-of-words, which is worth
knowing and worth reporting.

### Settings

- **Accelerator: GPU T4 x2** (or P100). Training on CPU is not viable here.
- **Internet: ON** - needed for `pip install` and the pretrained checkpoint.
- **Input:** add Notebook 03's committed output (*Add Input -> Notebook Output*).

*GPU quota is ~30 h/week. This notebook should cost well under an hour.*

---
## Pin the environment first

Hugging Face froze its TensorFlow support and removes it in `transformers` v5, and
TF ≥ 2.16 ships Keras 3, which the TF model classes do not support. Both have to be
handled **before** transformers is imported — a restart after installing is the reliable
way to guarantee that.

Run this cell, let the session restart if it does, then continue from the next cell.

In [ ]:
%pip install -q "transformers<5" "tf-keras" 2>/dev/null
print("installed - if the session restarts, just continue from the NEXT cell")

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # MUST precede the transformers import

import json, math, warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import transformers
from transformers import (DistilBertTokenizerFast,
                          TFDistilBertForSequenceClassification,
                          create_optimizer)
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
print("tensorflow  ", tf.__version__)
print("transformers", transformers.__version__)
assert transformers.__version__ < "5", "transformers v5 removed the TF classes"

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "No GPU - set Accelerator to GPU in notebook settings."
tf.keras.mixed_precision.set_global_policy("mixed_float16")   # ~1.7x faster on T4
print("mixed precision:", tf.keras.mixed_precision.global_policy().name)

---
## Load the splits

`issue` is a separate task with separate splits — this notebook never touches it. The
firewall below is Notebook 03's, unchanged: the test set raises rather than loading.

In [ ]:
TASK       = "product"
MAX_LENGTH = 256      # 25.0% truncation; 128 would truncate 53.8%
BATCH      = 32
EPOCHS     = 3
LR         = 3e-5
SEED       = 42

tf.keras.utils.set_random_seed(SEED)

def find_dir(name):
    for root, dirs, _ in os.walk("/kaggle/input"):
        if name in dirs:
            return os.path.join(root, name)
    return None

SPLITS = find_dir("splits")
assert SPLITS, "splits/ not found - add notebook 03's output via Add Input."
print("splits at:", SPLITS)

ALLOW_TEST = False    # this notebook must never load test

def load_split(task, name, base=SPLITS):
    if name == "test" and not ALLOW_TEST:
        raise RuntimeError(f"{task}/test is sealed until notebook 06.")
    return pd.read_csv(os.path.join(base, task, f"{name}.csv"))

train = load_split(TASK, "train")
val   = load_split(TASK, "val")
print(f"train {len(train)} | val {len(val)}")

try:
    load_split(TASK, "test"); print("FIREWALL BROKEN")
except RuntimeError as e:
    print("sealed:", e)

In [ ]:
# num_labels comes from label_maps.json, never a hardcoded number.
lm_path = None
for root, _, files in os.walk("/kaggle/input"):
    if "label_maps.json" in files:
        lm_path = os.path.join(root, "label_maps.json"); break
assert lm_path, "label_maps.json not found in notebook 03's output."

label_maps = json.load(open(lm_path))
CLASSES    = label_maps[TASK]["classes"]
NUM_LABELS = label_maps[TASK]["num_labels"]
id2label   = {i: c for i, c in enumerate(CLASSES)}
label2id   = {c: i for i, c in id2label.items()}

assert set(train["label"]) <= set(CLASSES), "unseen label in train"
assert NUM_LABELS == len(CLASSES) == train["label"].nunique()
print(f"num_labels = {NUM_LABELS}")
for i, c in id2label.items():
    print(f"  {i}: {c}")

---
## Tokenize

Train and validation only. Tokenizing the whole split at once is fine at this size and
keeps the `tf.data` pipeline simple.

In [ ]:
tok = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def make_ds(df, shuffle=False):
    enc = tok(df["text"].astype(str).tolist(),
              max_length=MAX_LENGTH, truncation=True,
              padding="max_length", return_tensors="np")
    y = df["label"].map(label2id).values.astype("int32")
    ds = tf.data.Dataset.from_tensor_slices((dict(enc), y))
    if shuffle:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train, shuffle=True)
val_ds   = make_ds(val)

xb, yb = next(iter(train_ds))
print("input_ids   ", xb["input_ids"].shape)
print("attention   ", xb["attention_mask"].shape)
print("labels      ", yb.shape, "| first 8:", yb[:8].numpy())
print("steps/epoch ", math.ceil(len(train) / BATCH))

---
## Class weights

The label distribution is skewed — for `product`, the largest class is ~18x the smallest.
Without weighting, the model can score well on accuracy while ignoring the tail, which is
exactly the failure macro-F1 exists to catch.

In [ ]:
y_train = train["label"].map(label2id).values
weights = compute_class_weight("balanced", classes=np.arange(NUM_LABELS), y=y_train)
class_weight = {i: float(w) for i, w in enumerate(weights)}

wdf = pd.DataFrame({
    "class":  [id2label[i] for i in range(NUM_LABELS)],
    "n":      [int((y_train == i).sum()) for i in range(NUM_LABELS)],
    "weight": [round(class_weight[i], 3) for i in range(NUM_LABELS)],
}).sort_values("n", ascending=False)
print(wdf.to_string(index=False))

---
## Macro-F1 checkpointing

Keras has no macro-F1 metric, so best-checkpoint selection is a hand-written callback:
predict on validation each epoch, compute `f1_score(average="macro")`, and call
`save_pretrained()` only on improvement.

This matters because Keras' own `ModelCheckpoint` can only track `val_loss` or
`val_accuracy` — and on skewed data neither is the quantity being optimised for.

In [ ]:
OUT = "/kaggle/working/best_model"

class MacroF1Checkpoint(tf.keras.callbacks.Callback):
    def __init__(self, ds, y_true, out_dir, tokenizer):
        super().__init__()
        self.ds, self.y_true = ds, y_true
        self.out_dir, self.tokenizer = out_dir, tokenizer
        self.best = -1.0
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        logits = self.model.predict(self.ds, verbose=0).logits
        pred = logits.argmax(-1)
        f1  = f1_score(self.y_true, pred, average="macro")
        acc = accuracy_score(self.y_true, pred)
        improved = f1 > self.best
        self.history.append({"epoch": epoch + 1, "val_macro_f1": round(float(f1), 4),
                             "val_accuracy": round(float(acc), 4),
                             "train_loss": round(float(logs.get("loss", 0)), 4),
                             "saved": bool(improved)})
        print(f"\n  epoch {epoch+1}: val macro-F1 = {f1:.4f} | val acc = {acc:.4f}"
              + ("  <- best, saving" if improved else ""))
        if improved:
            self.best = f1
            self.model.save_pretrained(self.out_dir)
            self.tokenizer.save_pretrained(self.out_dir)

y_val = val["label"].map(label2id).values
ckpt = MacroF1Checkpoint(val_ds, y_val, OUT, tok)
print("callback ready ->", OUT)

---
## Train

**`use_safetensors=False` is required, not optional.** Kaggle's `safetensors` is newer
than the pinned `transformers` expects: the TF loader runs `for key in pt_state_dict`
over what is now a non-iterable `safe_open` handle, and raises
`TypeError: 'builtins.safe_open' object is not iterable`. Skipping the safetensors path
loads `distilbert-base-uncased`'s native TF weights (`tf_model.h5`) instead — no
conversion, no torch involved.

`create_optimizer` gives AdamW with linear warmup and decay in one call. The loss is set
`from_logits=True` because the HF TF head returns raw logits.

In [ ]:
def load_tf_backbone():
    kw = dict(num_labels=NUM_LABELS, id2label=id2label, label2id=label2id)

    # 1. Native TF weights (tf_model.h5) - no torch, no conversion, no safetensors.
    try:
        m = TFDistilBertForSequenceClassification.from_pretrained(
            "distilbert-base-uncased", use_safetensors=False, **kw)
        print("loaded native TF weights (tf_model.h5)")
        return m
    except Exception as e:
        print(f"native TF load failed: {type(e).__name__}: {e}")

    # 2. Fall back to converting the PyTorch .bin (needs torch, which Kaggle has).
    m = TFDistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", from_pt=True, use_safetensors=False, **kw)
    print("loaded via from_pt conversion of pytorch_model.bin")
    return m

model = load_tf_backbone()

steps_per_epoch = math.ceil(len(train) / BATCH)
total_steps     = steps_per_epoch * EPOCHS

optimizer, schedule = create_optimizer(
    init_lr=LR,
    num_train_steps=total_steps,
    num_warmup_steps=int(0.1 * total_steps),
    weight_decay_rate=0.01,
)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
print(f"{total_steps} total steps | {steps_per_epoch} per epoch | warmup "
      f"{int(0.1*total_steps)}")

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=[ckpt],
    verbose=1,
)

---
## Results

Reloading the **saved best checkpoint** rather than scoring the final in-memory model —
if epoch 2 beat epoch 3, the in-memory weights are not the ones being shipped.

In [ ]:
best = TFDistilBertForSequenceClassification.from_pretrained(
    OUT, use_safetensors=False)          # same safetensors guard as the backbone load
logits = best.predict(val_ds, verbose=0).logits
pred = logits.argmax(-1)

f1  = f1_score(y_val, pred, average="macro")
acc = accuracy_score(y_val, pred)

BASE_TFIDF = 0.7748
BASE_DUMMY = 0.1103
PRIOR_3EP  = 0.7784

print(f"DistilBERT val macro-F1 = {f1:.4f}   accuracy = {acc:.4f}")
print(f"gap (acc - macro)       = {acc - f1:+.4f}\n")
print(f"{'stratified dummy':22s} {BASE_DUMMY:.4f}")
print(f"{'TF-IDF + LogReg':22s} {BASE_TFIDF:.4f}")
print(f"{'DistilBERT':22s} {f1:.4f}   ({f1 - BASE_TFIDF:+.4f} vs TF-IDF)")
print(f"{'  3-epoch run':22s} {PRIOR_3EP:.4f}   ({f1 - PRIOR_3EP:+.4f} from the longer schedule)")

if f1 > BASE_TFIDF:
    print("\n>>> DistilBERT wins. Report the margin honestly - a small win on a slow "
          "model is a finding about diminishing returns, not a triumph.")
else:
    print("\n>>> TF-IDF holds. This is a legitimate result: the lexical signal is "
          "already saturated by bag-of-words. Report it, do not bury it.")

print("\n" + classification_report(
    y_val, pred, target_names=[id2label[i] for i in range(NUM_LABELS)],
    zero_division=0))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

sns.set_theme(style="whitegrid")
hist = pd.DataFrame(ckpt.history)
display(hist)

fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))

ax[0].plot(hist["epoch"], hist["val_macro_f1"], marker="o", label="DistilBERT")
ax[0].axhline(BASE_TFIDF, ls="--", color="crimson", label=f"TF-IDF {BASE_TFIDF}")
ax[0].axhline(BASE_DUMMY, ls=":", color="grey", label=f"dummy {BASE_DUMMY}")
ax[0].set(title=f"{TASK}: validation macro-F1 by epoch",
          xlabel="epoch", ylabel="macro-F1")
ax[0].set_xticks(hist["epoch"])
ax[0].legend()

cm = confusion_matrix(y_val, pred, normalize="true")
short = [c[:26] + ("..." if len(c) > 26 else "") for c in
         (id2label[i] for i in range(NUM_LABELS))]
sns.heatmap(cm, annot=NUM_LABELS <= 10, fmt=".2f", cmap="Blues",
            xticklabels=short, yticklabels=short, ax=ax[1], cbar=False,
            vmin=0, vmax=1)
ax[1].set(title=f"{TASK}: row-normalised confusion", xlabel="predicted", ylabel="true")
plt.setp(ax[1].get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.setp(ax[1].get_yticklabels(), fontsize=8)

plt.tight_layout()
os.makedirs("/kaggle/working/plots", exist_ok=True)
plt.savefig(f"/kaggle/working/plots/07_distilbert_{TASK}.png", dpi=120,
            bbox_inches="tight")
plt.show()

---
## Is the margin real?

A single-seed difference of a few thousandths is not evidence of anything. The smallest
validation classes here hold only a few hundred rows, so a handful of examples flipping
moves macro-F1 by ~0.001-0.002 on its own.

This is a **paired bootstrap**: resample the validation set with replacement, score
*both* models on the same resample, and look at the distribution of the difference. If
the 95% interval spans zero, the two models are indistinguishable on this data and
claiming a win would be overreach.

**Be precise about what this does and does not cover.** The paired bootstrap measures
*evaluation* variance: would the margin survive a different draw of validation rows? It
does **not** measure *training* variance: would the margin survive a retrain with a
different seed?

Training variance is not small here. In a simulation of two classifiers with identical
error rates on this exact class-size profile, macro-F1 came out at 0.7322 and 0.7488 -
a spread of **0.0166** from nothing but different random errors. Any observed margin
smaller than that should be treated as a tie regardless of what the bootstrap says.

So: if the bootstrap interval spans zero, it is a tie. If it excludes zero but the
margin is under ~0.02, it is still probably a tie, and the way to settle it is to re-run
this notebook with a different `SEED` and compare - budget ~50 min per seed.

In [ ]:
from joblib import load as joblib_load

pipe_path = None
for root, _, files in os.walk("/kaggle/input"):
    if f"tfidf_lr_{TASK}.joblib" in files:
        pipe_path = os.path.join(root, f"tfidf_lr_{TASK}.joblib"); break

if pipe_path is None:
    print("TF-IDF pipeline not found in inputs - skipping the paired bootstrap.")
    boot = None
else:
    tfidf_pred = joblib_load(pipe_path).predict(val["text"])
    bert_pred  = np.array([id2label[i] for i in pred])
    y_true     = val["label"].values

    B, rng, n = 1000, np.random.default_rng(SEED), len(y_true)
    deltas = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, n)
        deltas[b] = (f1_score(y_true[idx], bert_pred[idx], average="macro")
                     - f1_score(y_true[idx], tfidf_pred[idx], average="macro"))

    lo, hi = np.percentile(deltas, [2.5, 97.5])
    p_win  = float((deltas > 0).mean())
    boot = {"n_resamples": B, "mean_delta": round(float(deltas.mean()), 4),
            "ci95_low": round(float(lo), 4), "ci95_high": round(float(hi), 4),
            "p_distilbert_better": round(p_win, 3),
            "distinguishable": bool(lo > 0 or hi < 0)}

    print(f"paired bootstrap over {B} resamples of the validation set")
    print(f"  mean delta (DistilBERT - TF-IDF) = {deltas.mean():+.4f}")
    print(f"  95% CI                           = [{lo:+.4f}, {hi:+.4f}]")
    print(f"  P(DistilBERT better)             = {p_win:.1%}")
    if lo > 0:
        print("\n>>> The interval excludes zero. DistilBERT is genuinely ahead.")
    elif hi < 0:
        print("\n>>> The interval excludes zero. TF-IDF is genuinely ahead.")
    else:
        print("\n>>> The interval SPANS ZERO. The two models are statistically")
        print("    indistinguishable on this validation set. Report a tie, not a win -")
        print("    and note that TF-IDF reaches it in seconds on CPU.")

    if abs(deltas.mean()) < 0.02:
        print("\n    NOTE: the margin is under 0.02, which is smaller than the")
        print("    training-variance spread expected from a reseed on this class-size")
        print("    profile (~0.017). Treat it as a tie whatever the interval says.")

In [ ]:
results = {
    "task": TASK,
    "num_labels": NUM_LABELS,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "class_weighted": True,
    "n_train": int(len(train)),
    "n_val": int(len(val)),
    "val_macro_f1": round(float(f1), 4),
    "val_accuracy": round(float(acc), 4),
    "baseline_tfidf": BASE_TFIDF,
    "baseline_dummy": BASE_DUMMY,
    "delta_vs_tfidf": round(float(f1 - BASE_TFIDF), 4),
    "beats_baseline": bool(f1 > BASE_TFIDF),
    "per_epoch": ckpt.history,
    "best_epoch": int(max(ckpt.history, key=lambda r: r["val_macro_f1"])["epoch"]),
    "still_improving_at_last_epoch": bool(
        len(ckpt.history) > 1
        and ckpt.history[-1]["val_macro_f1"] > ckpt.history[-2]["val_macro_f1"]),
    "paired_bootstrap": boot,
    "prior_3epoch_macro_f1": PRIOR_3EP,
    "delta_vs_3epoch": round(float(f1 - PRIOR_3EP), 4),
}
with open(f"/kaggle/working/distilbert_{TASK}_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))
print("\ncheckpoint files:", sorted(os.listdir(OUT)))

---
## Next

**Save Version -> Save & Run All.** Interactive `/kaggle/working/` does not survive the
session, so without a committed run there is no checkpoint for Notebook 06 to load.

The committed output carries:

- `best_model/` - the checkpoint from the best epoch, not the last
- `distilbert_product_results.json` - per-epoch history and the comparison
- `plots/07_distilbert_product.png`

**Notebook 05** repeats this for `issue` (15 labels). Same code, different task constant.

### If it underperforms

In order of expected value, not effort:

1. **Raise `MAX_LENGTH` to 384.** Truncation drops 25.0% -> 12.6%. The most likely real
   gain, at ~1.5x the training time.
2. **Train longer.** If validation macro-F1 was still climbing at the last epoch, the run
   was cut short, not converged.
3. **Lower the learning rate to 2e-5.** If the score moved erratically between epochs.
4. **Drop class weights** and compare. Weighting helps macro-F1 on skewed data in theory;
   verify it on this data rather than assuming.

Change one thing at a time and record each result — a tuning log is worth more in an
interview than a single unexplained number.